# 01c · SMLP4Rec with a query token (context-aware, no hybrid)

Notebook 01 feeds SMLP4Rec only the user's previous hotel clusters. Here the **current search** enters the model as one extra position (the query token). Plan: `docs/plans/plan_oct_07_query_token.md`.

- **History**: the same 20-cluster sequence as nb01 (RecBole right-padding), positions 0..19.
- **Query token** (position 20): learned `[MASK]` + the sum of one embedding per query field of the target booking's own search: destination (train vocabulary, count >= 5), destination type, check-in month, lead-time bucket, stay-length bucket, adults, children, rooms, package, mobile, channel, site, point-of-sale continent, user country (count >= 5). The hidden state at position 20 scores the 100 clusters.
- **Not used**: `destinations.csv` latent features, hotel fields (`hotel_*`), `orig_destination_distance`, `user_location_region/city`, `cnt`.
- **Same as nb01**: split (global temporal 80/10/10 over next-booking targets, warm rows only), config (2 layers, hidden 64, dropout 0, CE, Adam lr 1e-3, batch 1024, 3 epochs, seed 2022), full ranking over 100 clusters, Recall@K / NDCG@K for K = 5, 10, 20 (`nbp.eval.metrics`).
- **Not included**: prior, sameDest, hybrid, per-position context, cold (L = 0) users. One seed; 3 epochs is a pipeline-validation budget, not a tuned result.
- **Late-fusion check** (section 10): does adding the destination prior on top of this model still help?
- **Checks**: the plain model (nb01 `best.pth`) is rescored on exactly these rows and must reproduce nb01's test numbers, which proves the rows are the same. A query-shuffle and a history-mask diagnostic show what the model relies on.

Data: `scripts/expedia_query_to_recbole.py` writes `expedia_rowid.inter` (same rows as `expedia.inter` plus `row_id`) and `expedia_query_codes.npz`. Kernel: `smlp4rec`.

In [1]:
import functools
import json
import logging
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # import torch before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)  # RecBole checkpoints store the Config

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.data.query_features import FORBIDDEN_EXACT, FORBIDDEN_PREFIX, QUERY_FIELDS, RAW_COLS, build_vocab, encode
from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from nbp.models.smlprec_query import ROW_ID, SMLPRECQuery
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
CODES = ROOT / "data" / "interim" / "expedia_query_codes.npz"
PLAIN_RUN = ROOT / "experiments" / "2026-10-05_expedia_smlp4rec_week3-repro"
PLAIN_JSON = ROOT / "results" / "week3_implementation" / "smlprec_expedia_run.json"
RESULT_JSON = ROOT / "results" / "week4_rebuild" / "smlprec_query_run.json"
TAG = "query-1c"
RUN_DIR = ROOT / "experiments" / f"{date.today():%Y-%m-%d}_expedia_smlp4rec_{TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME = False  # True: continue from RUN_DIR/last.pth if it exists (weights + optimizer + per-epoch rows); set False or delete the run folder for a fresh run
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

# leak guard: no forbidden raw column feeds the query features
assert not [c for c in RAW_COLS if c in FORBIDDEN_EXACT or c.startswith(FORBIDDEN_PREFIX)]
print("query fields:", list(QUERY_FIELDS))

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_query-1c
query fields: ['dest', 'dest_type', 'ci_month', 'lead', 'stay', 'adults', 'children', 'rooms', 'package', 'mobile', 'channel', 'site', 'posa', 'country']


## 1. Config and data

Same RecBole config and seed as nb01; only the dataset differs (`expedia_rowid` = `expedia.inter` + `row_id`). RecBole copies `row_id` to every target row, which is how the model looks up the target's query features.

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_rowid",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "row_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])

t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
t_data = time.time() - t0

ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
IID, TIME, UID = config["ITEM_ID_FIELD"], config["TIME_FIELD"], config["USER_ID_FIELD"]


def split_inputs(loader) -> dict:
    # history ids (n, 20), history length (n,), target row id (n,), true next cluster (n,), user (n,)
    d = loader.dataset.inter_feat
    return {
        "seq": d[ITEM_SEQ],
        "length": d[ITEM_SEQ_LEN],
        "row": d[ROW_ID].long(),
        "y": d[IID].numpy(),
        "user": d[UID].numpy(),
    }


eval_inputs = {"valid": split_inputs(valid_data), "test": split_inputs(test_data)}
splits = {}
for name, loader in (("train", train_data), ("valid", valid_data), ("test", test_data)):
    d = loader.dataset.inter_feat
    ts = d[TIME].numpy()
    splits[name] = {
        "targets": len(ts),
        "first_target_unix": int(ts.min()),
        "last_target_unix": int(ts.max()),
        "mean_history_len": round(float(d[ITEM_SEQ_LEN].float().mean()), 3),
    }
print(f"data build {t_data:.1f}s; users {dataset.user_num - 1}, clusters {dataset.item_num - 1}")
print(json.dumps(splits, indent=1))

plain_ref = json.loads(PLAIN_JSON.read_text(encoding="utf-8"))
for name in ("train", "valid", "test"):
    for k in ("targets", "first_target_unix", "last_target_unix", "mean_history_len"):
        assert splits[name][k] == plain_ref["splits"][name][k], (name, k)
print("split sizes, time ranges and mean history length identical to nb01")

data build 191.4s; users 813985, clusters 100
{
 "train": {
  "targets": 1749368,
  "first_target_unix": 1357518976,
  "last_target_unix": 1412776704,
  "mean_history_len": 5.853
 },
 "valid": {
  "targets": 218670,
  "first_target_unix": 1412776704,
  "last_target_unix": 1416398080,
  "mean_history_len": 7.391
 },
 "test": {
  "targets": 218670,
  "first_target_unix": 1416398080,
  "last_target_unix": 1420070400,
  "mean_history_len": 7.255
 }
}
split sizes, time ranges and mean history length identical to nb01


## 2. Query features: train-only vocabulary and coverage

`codes[i]` holds the query codes of the booking with `row_id = i`. A code gets its own embedding row only if it appears often enough among the **train targets** (destination and user country: >= 5; other fields: >= 1); everything else (including values that appear only in valid/test) maps to index 0. Coverage = share of valid/test targets whose value is out of vocabulary.

In [3]:
import pandas as pd  # after torch

z = np.load(CODES)
codes, fields = z["codes"], list(z["fields"])
assert fields == list(QUERY_FIELDS)
# one code row per booking of the .inter file (row_id = line index)
assert codes.shape[0] == len(pd.read_csv(WORK / "expedia_rowid" / "expedia_rowid.inter", sep="	", usecols=["row_id:float"])), codes.shape

train_rows = train_data.dataset.inter_feat[ROW_ID].long().numpy()
vocab = build_vocab(codes, train_rows)
Q = encode(codes, vocab)  # (n_bookings, F) int32, 0 = out of vocabulary
field_sizes = [len(v) + 1 for v in vocab]

coverage = pd.DataFrame(
    {
        "vocab_size": [len(v) for v in vocab],
        "oov_valid_targets": [float((Q[eval_inputs["valid"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
        "oov_test_targets": [float((Q[eval_inputs["test"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
    },
    index=fields,
).round(4)
print(coverage)
np.savez_compressed(RUN_DIR / "vocab.npz", fields=np.array(fields), **{f"vocab_{f}": v for f, v in zip(fields, vocab)})

           vocab_size  oov_valid_targets  oov_test_targets
dest            13766             0.0227            0.0243
dest_type           8             0.0000            0.0000
ci_month           12             0.0000            0.0000
lead               11             0.0000            0.0000
stay               14             0.0000            0.0000
adults             10             0.0000            0.0000
children           10             0.0000            0.0000
rooms               9             0.0000            0.0000
package             2             0.0000            0.0000
mobile              2             0.0000            0.0000
channel            11             0.0000            0.0000
site               39             0.0003            0.0003
posa                5             0.0000            0.0000
country           212             0.0000            0.0000


## 3. Model and optimiser

`SMLPRECQuery` (`src/nbp/models/smlprec_query.py`): the legacy mixer block and item embedding plus one embedding table per query field. Optimiser as nb01: Adam, lr 1e-3, no weight decay, no gradient clipping.

In [4]:
Qt = torch.from_numpy(Q)
model = SMLPRECQuery(config, train_data.dataset, Qt, field_sizes).to(config["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
n_params = sum(p.numel() for p in model.parameters())
n_query_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("query_"))
print(model)
print(f"parameters: {n_params} (query token tables: {n_query_params}; history-only model nb01: {plain_ref['n_parameters']})")

SMLPRECQuery(
  (item_embedding): Embedding(101, 64, padding_idx=0)
  (layers): ModuleList(
    (0): SMLP(
      (c): Sequential(
        (0): Linear(in_features=1, out_features=3, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=3, out_features=1, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (h): Sequential(
        (0): Linear(in_features=21, out_features=63, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=63, out_features=21, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (w): Sequential(
        (0): Linear(in_features=64, out_features=192, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=192, out_features=64, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((64,), eps=1e-0

## 4. Training, scoring and evaluation functions

In [5]:
def train_epoch(model, loader, optimizer) -> float:
    # one epoch over the train loader; returns the sum of batch losses (as nb01 / RecBole)
    model.train()
    total = 0.0
    for inter in loader:
        inter = inter.to(config["device"])
        optimizer.zero_grad()
        loss = model.calculate_loss(inter)
        if torch.isnan(loss):
            raise ValueError("training loss is nan")
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total


@torch.no_grad()
def full_scores(model, inp: dict, q_idx=None, seq=None, batch: int = 8192) -> np.ndarray:
    # (n, n_items) scores of one split; padding column = -inf. q_idx / seq override the real inputs (diagnostics)
    model.eval()
    seq = inp["seq"] if seq is None else seq
    q_idx = Qt[inp["row"]] if q_idx is None else q_idx
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = model.scores(seq[i : i + batch], q_idx[i : i + batch].long())
        s[:, 0] = float("-inf")
        out.append(s.cpu().numpy())
    return np.concatenate(out)


def evaluate(model, inp: dict, **kw) -> tuple[dict, np.ndarray]:
    rank = target_rank(full_scores(model, inp, **kw), inp["y"])
    return summarize(rank, ks=KS), rank

## 5. Train (3 epochs)

Valid and test are scored after every epoch (test only for the record); best epoch = highest **valid NDCG@10**, strict improvement, as nb01. Checkpoints in the run folder: `epoch{n}.pth` (weights, every epoch), `last.pth` (weights + optimizer + per-epoch rows, for resuming with `RESUME = True`), and `best.pth` after training.

In [6]:
per_epoch, best, start = [], None, 1
if RESUME and (RUN_DIR / "last.pth").exists():
    ck = torch.load(RUN_DIR / "last.pth")
    model.load_state_dict(ck["state_dict"])
    optimizer.load_state_dict(ck["optimizer"])
    per_epoch, best, start = ck["per_epoch"], ck["best"], ck["epoch"] + 1
    print("resumed after epoch", ck["epoch"])

t1 = time.time()
for epoch in range(start, config["epochs"] + 1):
    t = time.time()
    loss = train_epoch(model, train_data, optimizer)
    row = {"epoch": epoch, "train_loss": round(loss, 4), "train_seconds": round(time.time() - t, 1)}
    t = time.time()
    ranks = {}
    for name, inp in eval_inputs.items():
        row[name], ranks[name] = evaluate(model, inp)
    row["eval_seconds"] = round(time.time() - t, 1)
    valid_score = ndcg_at_k(ranks["valid"], 10)
    torch.save(model.state_dict(), RUN_DIR / f"epoch{epoch}.pth")
    if best is None or valid_score > best["valid_score"]:
        best = {"epoch": epoch, "valid_score": valid_score}
    per_epoch.append(row)
    torch.save(
        {"epoch": epoch, "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
         "per_epoch": per_epoch, "best": best},
        RUN_DIR / "last.pth",
    )
    print(
        f"epoch {epoch}: loss {row['train_loss']:.4f} (train {row['train_seconds']}s, eval {row['eval_seconds']}s) | "
        f"valid R@5 {row['valid']['recall@5']:.4f} NDCG@10 {row['valid']['ndcg@10']:.4f} | "
        f"test R@5 {row['test']['recall@5']:.4f} NDCG@10 {row['test']['ndcg@10']:.4f}"
    )
t_fit = sum(r["train_seconds"] + r["eval_seconds"] for r in per_epoch)  # also correct after a resume
for r in per_epoch:
    print(
        f"epoch {r['epoch']}: loss {r['train_loss']:.4f} (train {r['train_seconds']}s, eval {r['eval_seconds']}s) | "
        f"valid R@5 {r['valid']['recall@5']:.4f} NDCG@10 {r['valid']['ndcg@10']:.4f} | "
        f"test R@5 {r['test']['recall@5']:.4f} NDCG@10 {r['test']['ndcg@10']:.4f}"
    )
print(f"train + per-epoch eval {t_fit:.1f}s; best epoch by valid NDCG@10: {best['epoch']}")

epoch 1: loss 5183.8896 (train 308.4s, eval 22.6s) | valid R@5 0.5832 NDCG@10 0.4716 | test R@5 0.5731 NDCG@10 0.4647


epoch 2: loss 4631.6362 (train 301.2s, eval 24.5s) | valid R@5 0.5965 NDCG@10 0.4828 | test R@5 0.5880 NDCG@10 0.4764


epoch 3: loss 4522.5900 (train 315.2s, eval 22.8s) | valid R@5 0.5981 NDCG@10 0.4832 | test R@5 0.5898 NDCG@10 0.4772
epoch 1: loss 5183.8896 (train 308.4s, eval 22.6s) | valid R@5 0.5832 NDCG@10 0.4716 | test R@5 0.5731 NDCG@10 0.4647
epoch 2: loss 4631.6362 (train 301.2s, eval 24.5s) | valid R@5 0.5965 NDCG@10 0.4828 | test R@5 0.5880 NDCG@10 0.4764
epoch 3: loss 4522.5900 (train 315.2s, eval 22.8s) | valid R@5 0.5981 NDCG@10 0.4832 | test R@5 0.5898 NDCG@10 0.4772
train + per-epoch eval 994.7s; best epoch by valid NDCG@10: 3


## 6. Best epoch: save `best.pth`, score valid and test once

`best.pth` holds the weights plus what is needed to rebuild the model without this notebook: `field_sizes`, `fields`, and the config keys that define the architecture. `vocab.npz` (same folder) maps raw codes to embedding indices; `row_id` lookups need `expedia_query_codes.npz`.

In [7]:
state = torch.load(RUN_DIR / f"epoch{best['epoch']}.pth")
model.load_state_dict(state)
ARCH_KEYS = ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob", "hidden_act",
             "layer_norm_eps", "initializer_range", "loss_type", "seed")
torch.save(
    {"epoch": best["epoch"], "state_dict": state, "field_sizes": field_sizes, "fields": fields,
     "config": {k: config[k] for k in ARCH_KEYS}},
    RUN_DIR / "best.pth",
)
t2 = time.time()
valid_best, rank_valid_q = evaluate(model, eval_inputs["valid"])
test_best, rank_test_q = evaluate(model, eval_inputs["test"])
t_test = time.time() - t2
print("valid:", valid_best)
print("test: ", test_best)

valid: {'n': 218670, 'recall@5': 0.5981, 'recall@10': 0.7725, 'recall@20': 0.9094, 'ndcg@5': 0.4266, 'ndcg@10': 0.4832, 'ndcg@20': 0.5181}
test:  {'n': 218670, 'recall@5': 0.5898, 'recall@10': 0.7636, 'recall@20': 0.9029, 'ndcg@5': 0.4209, 'ndcg@10': 0.4772, 'ndcg@20': 0.5127}


## 7. Reference rows on the same test rows: plain model, popularity, repeat-last

The plain (history-only) model is nb01's best checkpoint rescored on **these** rows. Its test numbers must equal nb01's (`results/week3_implementation/smlprec_expedia_run.json`); if they do, the rows, histories and targets are the same as in nb01.

In [8]:
plain = SMLPREC(config, train_data.dataset).to(config["device"])
plain.load_state_dict(torch.load(PLAIN_RUN / "best.pth")["state_dict"])
plain.eval()


@torch.no_grad()
def plain_rank(inp: dict, batch: int = 8192) -> np.ndarray:
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = plain.full_sort_predict({plain.ITEM_SEQ: inp["seq"][i : i + batch], plain.ITEM_SEQ_LEN: inp["length"][i : i + batch]})
        s[:, 0] = float("-inf")
        out.append(s.numpy())
    return target_rank(np.concatenate(out), inp["y"])


rank_valid_p, rank_test_p = plain_rank(eval_inputs["valid"]), plain_rank(eval_inputs["test"])
plain_test, plain_valid = summarize(rank_test_p, ks=KS), summarize(rank_valid_p, ks=KS)
for k, v in plain_ref["test"].items():
    if k in plain_test:
        assert abs(plain_test[k] - v) < 1e-4, (k, plain_test[k], v)
print("plain rescored on these rows reproduces nb01 test:", {k: plain_test[k] for k in ("recall@5", "ndcg@5", "ndcg@10")})

n_items = dataset.item_num
pop = np.bincount(train_data.dataset.inter_feat[IID].numpy(), minlength=n_items).astype(np.float64)
pop[0] = -np.inf
y = eval_inputs["test"]["y"]
seq_np, len_np = eval_inputs["test"]["seq"].numpy(), eval_inputs["test"]["length"].numpy()
last = seq_np[np.arange(len(y)), len_np - 1]
s_pop = np.tile(pop, (len(y), 1))
s_rep = s_pop.copy()
s_rep[np.arange(len(y)), last] = pop.max() + 1
heuristics = {
    "global_popularity": summarize(target_rank(s_pop, y), ks=KS),
    "repeat_last_then_popularity": summarize(target_rank(s_rep, y), ks=KS),
}
del s_pop, s_rep

plain rescored on these rows reproduces nb01 test: {'recall@5': 0.3369, 'ndcg@5': 0.2439, 'ndcg@10': 0.285}


## 8. Results (test, warm rows L >= 1, full ranking over 100 clusters)

In [9]:
COLS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
table = pd.DataFrame(
    {
        "global popularity": heuristics["global_popularity"],
        "repeat-last + popularity": heuristics["repeat_last_then_popularity"],
        "SMLP4Rec plain (history only, nb01)": plain_test,
        "SMLP4Rec + query token (1c)": test_best,
    }
).T[["n", *COLS]]
print(f"TEST (best epoch {best['epoch']} by valid NDCG@10)")
table

TEST (best epoch 3 by valid NDCG@10)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,218670.0,0.1478,0.0938,0.2476,0.1256,0.3925,0.1619
repeat-last + popularity,218670.0,0.2570,0.2031,0.3439,0.2309,0.4762,0.2641
"SMLP4Rec plain (history only, nb01)",218670.0,0.3369,0.2439,0.4645,0.2850,0.6309,0.3269
SMLP4Rec + query token (1c),218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127


### By user presence in train, and by history length

*Seen* = the user has at least one train target (as nb06). *Unseen, L >= 1* = users with earlier bookings but no train target. Cold users (L = 0) are not in these rows.

In [10]:
train_users = np.unique(train_data.dataset.inter_feat[UID].numpy())
seen = np.isin(eval_inputs["test"]["user"], train_users)
L = eval_inputs["test"]["length"].numpy()
masks = {
    "all warm rows": np.ones(len(L), bool),
    "seen users": seen,
    "unseen users (L>=1)": ~seen,
    "L=1": L == 1,
    "L=2-4": (L >= 2) & (L <= 4),
    "L=5-9": (L >= 5) & (L <= 9),
    "L>=10": L >= 10,
}
rows = []
for sl, m in masks.items():
    for name, r in (("plain", rank_test_p), ("query token", rank_test_q)):
        s = summarize(r[m], ks=(5,))
        rows.append({"slice": sl, "model": name, "n": s["n"], "recall@5": s["recall@5"], "ndcg@5": s["ndcg@5"]})
slices = pd.DataFrame(rows).set_index(["slice", "model"])
print(slices)

boot = {sl: paired_bootstrap(rank_test_q[m], rank_test_p[m], k=5) for sl, m in masks.items() if sl in ("all warm rows", "seen users", "unseen users (L>=1)")}
print("\nTEST, query token minus plain, paired bootstrap (1000 resamples, seed 0, 95% CI)")
pd.DataFrame(
    {sl: {m: f"{v[m]['diff']:+.4f} [{v[m]['ci95'][0]:+.4f}, {v[m]['ci95'][1]:+.4f}]" for m in ("recall@5", "ndcg@5")} for sl, v in boot.items()}
).T

                                      n  recall@5  ndcg@5
slice               model                                
all warm rows       plain        218670    0.3369  0.2439
                    query token  218670    0.5898  0.4209
seen users          plain        143025    0.3406  0.2433
                    query token  143025    0.5997  0.4282
unseen users (L>=1) plain         75645    0.3299  0.2451
                    query token   75645    0.5710  0.4071
L=1                 plain         39993    0.2691  0.1971
                    query token   39993    0.5549  0.3891
L=2-4               plain         68253    0.3155  0.2275
                    query token   68253    0.5771  0.4097
L=5-9               plain         47412    0.3551  0.2556
                    query token   47412    0.5997  0.4293
L>=10               plain         63012    0.3893  0.2826
                    query token   63012    0.6182  0.4467



TEST, query token minus plain, paired bootstrap (1000 resamples, seed 0, 95% CI)


,recall@5,ndcg@5
all warm rows,"+0.2529 [+0.2504, +0.2551]","+0.1770 [+0.1752, +0.1785]"
seen users,"+0.2591 [+0.2561, +0.2621]","+0.1849 [+0.1827, +0.1870]"
unseen users (L>=1),"+0.2412 [+0.2374, +0.2450]","+0.1619 [+0.1593, +0.1647]"


## 9. Diagnostics (same checkpoint, no training, no tuning)

- **Query shuffled**: each test row gets another test row's query features (history unchanged). If the model uses the query, scores fall.
- **History masked**: the history is replaced by padding (query unchanged). Shows what the query alone carries.
- **Both**: no information about the row.

In [11]:
inp = eval_inputs["test"]
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(inp["y"]), generator=g)
q_real = Qt[inp["row"]]
zeros_seq = torch.zeros_like(inp["seq"])
diag = {
    "real": test_best,
    "query shuffled": evaluate(model, inp, q_idx=q_real[perm])[0],
    "history masked": evaluate(model, inp, seq=zeros_seq)[0],
    "both (shuffled + masked)": evaluate(model, inp, q_idx=q_real[perm], seq=zeros_seq)[0],
}
print("TEST diagnostics, SMLP4Rec + query token")
pd.DataFrame(diag).T[["n", *COLS]]

TEST diagnostics, SMLP4Rec + query token


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
real,218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127
query shuffled,218670.0,0.1315,0.0864,0.2203,0.1148,0.3699,0.1523
history masked,218670.0,0.5165,0.3496,0.7090,0.4120,0.8741,0.4540
both (shuffled + masked),218670.0,0.0774,0.0462,0.1476,0.0686,0.2772,0.1011


## 10. Does late fusion with the destination prior still help? (hypothesis check)

Hypothesis: the query token already carries what the destination prior carries, so adding the prior on top should add little. Test: nb03's late fusion applied to **this** model, `score = log p_query-token + w * log p_prior(cluster | searched destination)`.

- **Prior** (as nb03): destination cluster counts smoothed to the destination's market (`m = 5`), unseen destination -> global; built only from bookings before the last train target (minus 128 s, float32 time resolution), so no valid/test booking feeds its own prior.
- **Weight** `w`: one global value chosen on **valid** by Recall@5 (ties -> smaller `w`); test read once. `w = 0` must reproduce the model alone.
- **Read-out**: test overall, paired bootstrap of the difference, and slices by how often the destination appears in the prior data and by whether the destination is out of the query-token vocabulary (where the model has no destination embedding).

In [12]:
TIME_MARGIN, N_CLUSTERS, M_SMOOTH, EPS = 128, 100, 5, 1e-6
W_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]
RAW = ROOT / "data" / "raw" / "hospitality" / "expedia" / "train.csv"
DEST = fields.index("dest")
tok2id = dataset.field2token_id[dataset.iid_field]
cluster_ids = np.array([tok2id[str(k)] for k in range(N_CLUSTERS)])  # model column of each raw cluster
n_items = dataset.item_num
cut = float(train_data.dataset.inter_feat[TIME].max()) - TIME_MARGIN
COLS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]


def build_prior_tables(cut_seconds: float) -> dict:
    # cluster count tables from bookings with unix time < cut_seconds (one chunked pass over train.csv)
    cols = ["date_time", "is_booking", "srch_destination_id", "hotel_market", "hotel_cluster"]
    dtypes = {c: "int32" for c in cols if c != "date_time"}
    parts = []
    for ch in pd.read_csv(RAW, usecols=cols, chunksize=2_000_000, dtype=dtypes):
        ch = ch[ch.is_booking == 1]
        ts = (pd.to_datetime(ch["date_time"]) - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)
        parts.append(ch.loc[ts.values < cut_seconds, ["srch_destination_id", "hotel_market", "hotel_cluster"]])
    b = pd.concat(parts, ignore_index=True)

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    # p(cluster | destination), shape (100,), raw cluster order
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_logp(dests: np.ndarray) -> np.ndarray:
    # (n, n_items) log prior in model-id columns; padding column 0 stays 0 (masked at ranking)
    uniq, inv = np.unique(dests, return_inverse=True)
    tab = np.stack([np.log(prior_vector(int(d)) + EPS) for d in uniq]).astype(np.float32)
    out = np.zeros((len(dests), n_items), np.float32)
    out[:, cluster_ids] = tab[inv]
    return out


t3 = time.time()
tables = build_prior_tables(cut)
print(f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t3:.0f}s")

q_dest = {sp: codes[inp["row"].numpy(), DEST].astype(np.int64) for sp, inp in eval_inputs.items()}
pr = {sp: prior_logp(q_dest[sp]) for sp in eval_inputs}
ml = {}
for sp, inp in eval_inputs.items():
    ml[sp] = torch.log_softmax(torch.from_numpy(full_scores(model, inp)), dim=1).numpy()


def rank_rows(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores = scores.copy()
    scores[:, 0] = -np.inf
    return target_rank(scores, y)


# w = 0 must reproduce the model alone
got = summarize(rank_rows(ml["test"], eval_inputs["test"]["y"]), ks=KS)
assert abs(got["recall@5"] - test_best["recall@5"]) < 1e-4 and abs(got["ndcg@5"] - test_best["ndcg@5"]) < 1e-4
print("w = 0 reproduces the query-token model: test Recall@5", got["recall@5"], "NDCG@5", got["ndcg@5"])

prior cut (unix s) 1412776576; 2463082 bookings, 34506 destinations; 61s


w = 0 reproduces the query-token model: test Recall@5 0.5898 NDCG@5 0.4209


In [13]:
sweep_valid = {}
for w in W_GRID:
    s = summarize(rank_rows(ml["valid"] + w * pr["valid"], eval_inputs["valid"]["y"]), ks=KS)
    sweep_valid[w] = {"recall@5": s["recall@5"], "ndcg@5": s["ndcg@5"]}
best_w = max(W_GRID, key=lambda w: (sweep_valid[w]["recall@5"], -w))
print("VALID sweep of w (w = 0 is the model alone):")
print(pd.DataFrame(sweep_valid).T)
print("chosen w on valid (Recall@5, ties -> smaller):", best_w)

yt = eval_inputs["test"]["y"]
rk = {
    "query token alone (w=0)": rank_rows(ml["test"], yt),
    f"query token + prior (w={best_w})": rank_rows(ml["test"] + best_w * pr["test"], yt),
    "destination prior only": rank_rows(pr["test"] + 1e-6 * np.nan_to_num(ml["test"], neginf=0.0), yt),
}
fusion_test = {k: summarize(v, ks=KS) for k, v in rk.items()}
print("\nTEST, warm rows")
pd.DataFrame(fusion_test).T[["n", *COLS]]

VALID sweep of w (w = 0 is the model alone):
      recall@5  ndcg@5
0.00    0.5981  0.4266
0.10    0.6020  0.4290
0.25    0.6039  0.4300
0.50    0.6025  0.4287
0.75    0.6002  0.4264
1.00    0.5970  0.4234
1.50    0.5910  0.4175
2.00    0.5858  0.4122
3.00    0.5776  0.4042
5.00    0.5679  0.3947
chosen w on valid (Recall@5, ties -> smaller): 0.25



TEST, warm rows


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
query token alone (w=0),218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127
query token + prior (w=0.25),218670.0,0.5942,0.4235,0.7710,0.4808,0.9125,0.5168
destination prior only,218670.0,0.5298,0.3631,0.7213,0.4251,0.8849,0.4666


In [14]:
A, B = f"query token + prior (w={best_w})", "query token alone (w=0)"
boot_fusion = paired_bootstrap(rk[A], rk[B], k=5)
print("TEST, (query token + prior) minus query token alone, paired bootstrap (1000 resamples, seed 0, 95% CI)")
print({m: f"{v['diff']:+.4f} [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]" for m, v in boot_fusion.items()})

dest_n = np.array([tables["dest"][int(d)].sum() if int(d) in tables["dest"] else 0 for d in q_dest["test"]])
oov_dest = Q[eval_inputs["test"]["row"].numpy(), DEST] == 0
slice_masks = {
    "destination unseen in prior data": dest_n == 0,
    "destination 1-19 bookings in prior data": (dest_n >= 1) & (dest_n < 20),
    "destination 20-199": (dest_n >= 20) & (dest_n < 200),
    "destination 200+": dest_n >= 200,
    "destination out of query-token vocab": oov_dest,
    "destination in query-token vocab": ~oov_dest,
}
rows = []
for sl, m in slice_masks.items():
    for name, r in (("query token alone", rk[B]), ("query token + prior", rk[A]), ("prior only", rk["destination prior only"])):
        s = summarize(r[m], ks=(5,))
        rows.append({"slice": sl, "model": name, "n": s["n"], "recall@5": s["recall@5"], "ndcg@5": s["ndcg@5"]})
slice_fusion = pd.DataFrame(rows).set_index(["slice", "model"])
print("\nTEST slices (Recall@5 / NDCG@5)")
slice_fusion

TEST, (query token + prior) minus query token alone, paired bootstrap (1000 resamples, seed 0, 95% CI)
{'recall@5': '+0.0045 [+0.0037, +0.0053]', 'ndcg@5': '+0.0026 [+0.0021, +0.0031]'}



TEST slices (Recall@5 / NDCG@5)


n  recall@5  \
slice                                   model                                   
destination unseen in prior data        query token alone      1503    0.3380   
                                        query token + prior    1503    0.3293   
                                        prior only             1503    0.1071   
destination 1-19 bookings in prior data query token alone      9950    0.5753   
                                        query token + prior    9950    0.6508   
                                        prior only             9950    0.6818   
destination 20-199                      query token alone     34761    0.7165   
                                        query token + prior   34761    0.7218   
                                        prior only            34761    0.6904   
destination 200+                        query token alone    172456    0.5673   
                                        query token + prior  172456    0.5676   
                                        prior only           172456    0.4923   
destination out of query-token vocab    query token alone      5323    0.3483   
                                        query token + prior    5323    0.4685   
                                        prior only             5323    0.4730   
destination in query-token vocab        query token alone    213347    0.5958   
                                        query token + prior  213347    0.5974   
                                        prior only           213347    0.5312   

                                                             ndcg@5  
slice                                   model                        
destination unseen in prior data        query token alone    0.2315  
                                        query token + prior  0.2248  
                                        prior only           0.0783  
destination 1-19 bookings in prior data query token alone    0.4371  
                                        query token + prior  0.4908  
                                        prior only           0.5259  
destination 20-199                      query token alone    0.5231  
                                        query token + prior  0.5261  
                                        prior only           0.4883  
destination 200+                        query token alone    0.4010  
                                        query token + prior  0.4006  
                                        prior only           0.3310  
destination out of query-token vocab    query token alone    0.2400  
                                        query token + prior  0.3294  
                                        prior only           0.3704  
destination in query-token vocab        query token alone    0.4254  
                                        query token + prior  0.4258  
                                        prior only           0.3629

## 5. Query token + sameDest (mô hình cuối), so với hybrid cũ (notebook 04)

**Hybrid cũ** = SMLP4Rec plain + prior điểm đến + sameDest (notebook 04). Ở đây thay SMLP4Rec plain bằng mô hình có query token ở trên, **bỏ prior** (mục 10 cho thấy query token đã chứa tín hiệu điểm đến mà prior mang), giữ nguyên phần còn lại: công thức `score = log q_model + w_s * log q_sameDest` với `log q = log((1 - alpha) * p + alpha / K)`, lưới `(alpha, w_s)`, quy tắc một sai số chuẩn trên valid, và 2 bộ trọng số (điểm đến mới / đã đặt trong lịch sử, đúng phương án notebook 04 đã chọn).

Biến thể duy nhất: **query token + sameDest** (w_p = 0). Biến thể có prior (query token + prior + sameDest) đã bị loại khỏi notebook.

Dòng L = 0 (lần đặt đầu): mô hình chạy thẳng với lịch sử rỗng; sameDest tự tắt (phân phối đều) nên dòng này dùng bộ trọng số "điểm đến mới". Hybrid cũ giữ định nghĩa của notebook 04: L = 0 chỉ dùng prior. Hybrid cũ được tính lại từ checkpoint plain và trọng số đã lưu, phải tái tạo số của notebook 04.


In [15]:
from nbp.hybrid.fusion import one_se_pick, same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores

HY_REF = json.loads((ROOT / "results" / "week3_implementation" / "smlprec_expedia_hybrid_samedest.json").read_text(encoding="utf-8"))
LF_REF = json.loads((ROOT / "results" / "week3_implementation" / "smlprec_expedia_late_fusion.json").read_text(encoding="utf-8"))
assert HY_REF["cohorts"]["chosen_fewest_cells_within_one_se"] == "known vs new destination (2)"
OLD_W = {int(k): tuple(v) for k, v in HY_REF["cohorts"]["final_weights_per_cell"].items()}  # cell 0 = new destination, 1 = known
OLD_ALPHA = HY_REF["selection"]["one_se_pick"][0]
OLD_NAME = "SMLP4Rec + prior + sameDest (this notebook)"
FUSION_W = LF_REF["best_global_w_on_valid"]
RECENCY = 0.7
ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5]
W_GRID2 = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
LS = config["LIST_SUFFIX"]
col_of_model_id = {int(m): k for k, m in enumerate(cluster_ids)}
Y = {sp: np.array([col_of_model_id[int(t)] for t in eval_inputs[sp]["y"]]) for sp in eval_inputs}
UV = eval_inputs["valid"]["user"]


def prior_prob(dests: np.ndarray) -> np.ndarray:
    uniq, inv = np.unique(dests, return_inverse=True)
    return np.stack([prior_vector(int(d)) for d in uniq]).astype(np.float32)[inv]


@torch.no_grad()
def plain_prob(inp: dict, batch: int = 8192) -> np.ndarray:
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = plain.full_sort_predict({plain.ITEM_SEQ: inp["seq"][i : i + batch], plain.ITEM_SEQ_LEN: inp["length"][i : i + batch]})
        s[:, 0] = float("-inf")
        out.append(torch.softmax(s, dim=1).numpy()[:, cluster_ids])
    return np.concatenate(out)


C = {}
for sp, loader in (("valid", valid_data), ("test", test_data)):
    inp = eval_inputs[sp]
    h_row = loader.dataset.inter_feat[ROW_ID + LS].long().numpy()
    seq, length = inp["seq"].numpy(), inp["length"].numpy()
    h_dest = codes[h_row, DEST].astype(np.int64)  # (n, S); positions >= length are ignored
    q_dest = codes[inp["row"].numpy(), DEST].astype(np.int64)
    pos = np.arange(seq.shape[1])[None, :] < length[:, None]
    C[sp] = {
        "y": Y[sp],
        "q_seen": ((h_dest == q_dest[:, None]) & pos).any(1),
        "prior": prior_prob(q_dest),
        "sd": same_dest_scores(seq, length, h_dest, q_dest, n_items, base=RECENCY, normalize=True)[:, cluster_ids],
        "p_plain": plain_prob(inp),
        "p_qt": np.exp(ml[sp][:, cluster_ids]),
    }
print({sp: (len(c["y"]), round(float(c["q_seen"].mean()), 4)) for sp, c in C.items()}, "rows, share of rows with the query destination in the history")


{'valid': (218670, 0.2999), 'test': (218670, 0.2991)} rows, share of rows with the query destination in the history


In [16]:
def rank_hybrid(c: dict, p_base: np.ndarray, alpha: float, wmap: dict) -> np.ndarray:
    """Rank of the target under log q_base + w_p log q_prior + w_s log q_sameDest; wmap = {0: (w_p, w_s) new dest, 1: known dest}."""
    cid = c["q_seen"].astype(int)
    wp = np.array([wmap[0][0], wmap[1][0]], np.float32)[cid][:, None]
    ws = np.array([wmap[0][1], wmap[1][1]], np.float32)[cid][:, None]
    return target_rank(
        smoothed_log(p_base, alpha) + wp * smoothed_log(c["prior"], alpha) + ws * smoothed_log(same_dest_distribution(c["sd"]), alpha),
        c["y"],
    )


# the old hybrid must be reproduced from the plain checkpoint and the saved weights (notebook 04)
old_wmap = {k: (v[1], v[2]) for k, v in OLD_W.items()}
rank_old_test = rank_hybrid(C["test"], C["test"]["p_plain"], OLD_ALPHA, old_wmap)
got, want = summarize(rank_old_test, ks=KS), HY_REF["benchmark"]["test"]["warm"][OLD_NAME]
print("old hybrid, test warm, recomputed:", {m: got[m] for m in ("recall@5", "ndcg@5", "ndcg@10")}, "| notebook 04:", {m: want[m] for m in ("recall@5", "ndcg@5", "ndcg@10")})
assert all(abs(got[m] - want[m]) < 1e-4 for m in ("recall@5", "ndcg@5", "recall@10", "ndcg@10"))


old hybrid, test warm, recomputed: {'recall@5': 0.6106, 'ndcg@5': 0.4539, 'ndcg@10': 0.507} | notebook 04: {'recall@5': 0.6106, 'ndcg@5': 0.4539, 'ndcg@10': 0.507}


In [17]:
def joint(r: np.ndarray) -> np.ndarray:
    """Per-row Recall@5 + NDCG@5 (float32), as notebook 04."""
    r = r.astype(np.float32)
    return ((r <= 5) * (1.0 + 1.0 / np.log2(r + 1))).astype(np.float32)


def complexity(c: tuple) -> tuple:
    return (max(c[1], c[2]), c[1] + c[2], -c[0])


def tune(p_base_valid: np.ndarray, fix_wp_zero: bool) -> dict:
    """(alpha, w_p, w_s) grid on valid -> alpha by the one-SE rule, then one weight pair per cell (new / known destination)."""
    cfgs = [(a, wp, ws) for a in ALPHAS for wp in W_GRID2 for ws in W_GRID2 if not (fix_wp_zero and wp != 0.0)]
    v = C["valid"]
    grid = {}
    for a in ALPHAS:
        M = smoothed_log(p_base_valid, a)
        P = smoothed_log(v["prior"], a)
        S = smoothed_log(same_dest_distribution(v["sd"]), a)
        for c in (c for c in cfgs if c[0] == a):
            grid[c] = target_rank(M + c[1] * P + c[2] * S, v["y"]).astype(np.int16)
    pick, best, eligible = one_se_pick({c: joint(r) for c, r in grid.items()}, UV, complexity)
    alpha = pick[0]
    cfg_a = [c for c in grid if c[0] == alpha]
    cell = v["q_seen"].astype(int)
    wmap = {}
    for cid in (0, 1):
        m = cell == cid
        cand = cfg_a if v["q_seen"][m].any() else [c for c in cfg_a if c[2] == 0.0]
        bc = one_se_pick({c: joint(grid[c][m]) for c in cand}, UV[m], complexity)[0]
        wmap[cid] = (bc[1], bc[2])
    return {"n_configs": len(cfgs), "best_on_valid": list(best), "one_se_pick": list(pick), "n_within_one_se": len(eligible), "alpha": alpha, "weights": wmap}


t5 = time.time()
tuned = {"A": tune(C["valid"]["p_qt"], fix_wp_zero=True)}
print(f"tuned on valid in {time.time() - t5:.0f}s")
for k, t in tuned.items():
    print(k, "| alpha", t["alpha"], "| weights per cell (w_p, w_s), 0 = destination new to the user, 1 = known:", t["weights"], f"| {t['n_within_one_se']} of {t['n_configs']} configurations within one SE")


tuned on valid in 26s
A | alpha 0.5 | weights per cell (w_p, w_s), 0 = destination new to the user, 1 = known: {0: (0.0, 0.0), 1: (0.0, 1.0)} | 10 of 50 configurations within one SE


In [18]:
# L = 0 rows of the test window (first booking of each user): query-token variants run on an empty history, as scripts/query_token_cold_users.py
inter_ids = pd.read_csv(WORK / "expedia_rowid" / "expedia_rowid.inter", sep="\t").sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter_ids.drop_duplicates("user_id:token", keep="first")
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
f = first[first["timestamp:float"] >= t_valid_max]
cold_row = f["row_id:float"].astype(int).values
cold_y = f["item_id:token"].astype(int).values  # raw cluster index 0..99
cold_dest = codes[cold_row, DEST].astype(np.int64)
S_LEN = config["MAX_ITEM_LIST_LENGTH"]
ml_cold = torch.log_softmax(
    torch.from_numpy(full_scores(model, {"y": cold_y}, q_idx=Qt[cold_row], seq=torch.zeros((len(cold_y), S_LEN), dtype=torch.long))), dim=1
).numpy()
C["cold"] = {
    "y": cold_y,
    "q_seen": np.zeros(len(cold_y), bool),
    "prior": prior_prob(cold_dest),
    "sd": np.zeros((len(cold_y), len(cluster_ids)), np.float32),  # no history: sameDest is uniform after normalisation
    "p_qt": np.exp(ml_cold[:, cluster_ids]),
}
assert len(cold_y) == 49443, len(cold_y)  # same rows as notebook 06

NAMES = {
    "plain": "SMLP4Rec plain",
    "plain+prior": "SMLP4Rec plain + prior (nb 03)",
    "old": "Hybrid cũ: plain + prior + sameDest (nb 04)",
    "qt": "Query token",
    "qt+prior": f"Query token + prior (w = {best_w})",
    "A": "Query token + sameDest",
}


def ranks_for(part: str) -> dict:
    c = C[part]
    r = {}
    r[NAMES["qt"]] = target_rank(np.log(c["p_qt"] + 1e-30), c["y"])
    r[NAMES["qt+prior"]] = target_rank(np.log(c["p_qt"] + 1e-30) + best_w * np.log(c["prior"] + EPS), c["y"])
    r[NAMES["A"]] = rank_hybrid(c, c["p_qt"], tuned["A"]["alpha"], tuned["A"]["weights"])
    if part == "test":
        r[NAMES["plain"]] = target_rank(np.log(c["p_plain"] + 1e-30), c["y"])
        r[NAMES["plain+prior"]] = target_rank(np.log(c["p_plain"] + 1e-30) + FUSION_W * np.log(c["prior"] + EPS), c["y"])
        r[NAMES["old"]] = rank_old_test
    else:  # cold rows: plain = global popularity; plain + prior and the old hybrid = prior only (definition of notebooks 03 / 04 / 06)
        pr_only = target_rank(np.log(c["prior"] + EPS), c["y"])
        r[NAMES["plain"]] = target_rank(np.tile(pop[cluster_ids], (len(c["y"]), 1)), c["y"])
        r[NAMES["plain+prior"]] = pr_only
        r[NAMES["old"]] = pr_only
    return r


R_warm, R_cold = ranks_for("test"), ranks_for("cold")
ORDER = [NAMES[k] for k in ("plain", "plain+prior", "old", "qt", "A")]
bench = {
    "warm": {n: summarize(R_warm[n], ks=KS) for n in ORDER},
    "all_events": {n: summarize(np.concatenate([R_warm[n], R_cold[n]]), ks=KS) for n in ORDER},
}
# cross-check against the saved all-events result of notebook 04
chk = HY_REF["benchmark"]["test"]["all_events"][OLD_NAME]
assert abs(bench["all_events"][NAMES["old"]]["ndcg@5"] - chk["ndcg@5"]) < 2e-3, (bench["all_events"][NAMES["old"]]["ndcg@5"], chk["ndcg@5"])
for scope in ("warm", "all_events"):
    print(f"TEST, {scope}" + (" (user có lịch sử, L >= 1)" if scope == "warm" else " (thêm 49.443 dòng L = 0)"))
    display(pd.DataFrame(bench[scope]).T[["n", *COLS]])


TEST, warm (user có lịch sử, L >= 1)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
SMLP4Rec plain,218670.0,0.3369,0.2439,0.4645,0.2850,0.6309,0.3269
SMLP4Rec plain + prior (nb 03),218670.0,0.5919,0.4227,0.7657,0.4791,0.9086,0.5154
Hybrid cũ: plain + prior + sameDest (nb 04),218670.0,0.6106,0.4539,0.7741,0.5070,0.9090,0.5413
Query token,218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127
Query token + sameDest,218670.0,0.6060,0.4475,0.7709,0.5009,0.9051,0.5351


TEST, all_events (thêm 49.443 dòng L = 0)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
SMLP4Rec plain,268113.0,0.2990,0.2139,0.4206,0.2530,0.5807,0.2932
SMLP4Rec plain + prior (nb 03),268113.0,0.5772,0.4094,0.7539,0.4667,0.9024,0.5045
Hybrid cũ: plain + prior + sameDest (nb 04),268113.0,0.5925,0.4349,0.7608,0.4894,0.9027,0.5255
Query token,268113.0,0.5736,0.4063,0.7505,0.4637,0.8956,0.5006
Query token + sameDest,268113.0,0.5868,0.4280,0.7565,0.4830,0.8974,0.5188


In [19]:
old_n = NAMES["old"]
boot5 = {}
for key in ("A", "qt"):
    n = NAMES[key]
    boot5[f"{n} minus {old_n}"] = {
        "warm": paired_bootstrap(R_warm[n], R_warm[old_n], k=5),
        "all_events": paired_bootstrap(np.concatenate([R_warm[n], R_cold[n]]), np.concatenate([R_warm[old_n], R_cold[old_n]]), k=5),
    }
print("paired bootstrap, test (1000 resamples, seed 0, 95% CI)")
display(
    pd.DataFrame(
        {
            (k, scope): {m: f"{v[scope][m]['diff']:+.4f} [{v[scope][m]['ci95'][0]:+.4f}, {v[scope][m]['ci95'][1]:+.4f}]" for m in ("recall@5", "ndcg@5")}
            for k, v in boot5.items()
            for scope in v
        }
    ).T
)

qs = C["test"]["q_seen"]
slice5 = {}
for sl, m in (("known destination", qs), ("new destination", ~qs)):
    cells = {}
    for n in ORDER:
        s = summarize(R_warm[n][m], ks=(5,))
        cells[n] = f"{s['recall@5']:.4f} / {s['ndcg@5']:.4f}"
    slice5[sl] = cells | {"rows": int(m.sum())}
print("TEST warm, slices by whether the query destination is in the history, Recall@5 / NDCG@5")
display(pd.DataFrame(slice5).T)

samedest_result = {
    "description": "query-token model + sameDest (A; the variant with prior was dropped), same formula / grid / one-SE rule as notebook 04, compared with the old hybrid (plain + prior + sameDest) recomputed from the nb04 weights",
    "formula": "score = log q_model + w_s * log q_sameDest; log q = log((1 - alpha) * p + alpha / 100); cells: query destination new / already in the history",
    "variants": {"A": "query token + sameDest (w_p = 0)"},
    "grid": {"alpha": ALPHAS, "w_s": W_GRID2},
    "tuned_on_valid": {k: {**t, "weights": {str(c): list(w) for c, w in t["weights"].items()}} for k, t in tuned.items()},
    "old_hybrid_weights_nb04": {"alpha": OLD_ALPHA, "weights": {str(c): list(w) for c, w in OLD_W.items()}},
    "cold_rows": "L = 0 (49,443 test first bookings): query-token model on an empty history with the 'new destination' weights; plain = popularity, plain + prior and the old hybrid = prior only",
    "benchmark_test": bench,
    "bootstrap_test_95ci": boot5,
    "test_slices_recall@5_ndcg@5": slice5,
}


paired bootstrap, test (1000 resamples, seed 0, 95% CI)


recall@5  \
Query token + sameDest minus Hybrid cũ: plain +... warm        -0.0046 [-0.0061, -0.0032]   
                                                   all_events  -0.0057 [-0.0070, -0.0044]   
Query token minus Hybrid cũ: plain + prior + sa... warm        -0.0209 [-0.0223, -0.0194]   
                                                   all_events  -0.0189 [-0.0204, -0.0175]   

                                                                                   ndcg@5  
Query token + sameDest minus Hybrid cũ: plain +... warm        -0.0065 [-0.0074, -0.0056]  
                                                   all_events  -0.0069 [-0.0077, -0.0060]  
Query token minus Hybrid cũ: plain + prior + sa... warm        -0.0331 [-0.0343, -0.0320]  
                                                   all_events  -0.0286 [-0.0297, -0.0275]

TEST warm, slices by whether the query destination is in the history, Recall@5 / NDCG@5


,SMLP4Rec plain,SMLP4Rec plain + prior (nb 03),Hybrid cũ: plain + prior + sameDest (nb 04),Query token,Query token + sameDest,rows
known destination,0.5968 / 0.4815,0.6785 / 0.5188,0.7401 / 0.6175,0.6870 / 0.5293,0.7413 / 0.6182,65407
new destination,0.2259 / 0.1425,0.5550 / 0.3817,0.5554 / 0.3842,0.5483 / 0.3746,0.5483 / 0.3746,153263


## 11. Save `smlprec_query_run.json` and `metrics.json`

Every number later quoted in slides or reports comes from this file.

In [20]:
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()
result = {
    "run": "SMLP4Rec + query token (context-aware, no hybrid), 3 epochs, notebook 01c",
    "dataset": "expedia (train.csv, is_booking == 1; item = hotel_cluster)",
    "git_commit": commit,
    "query_fields": fields,
    "query_vocab_min_count": dict(QUERY_FIELDS),
    "excluded_inputs": "destinations.csv latent features, hotel_*, orig_destination_distance, user_location_region/city, cnt",
    "config": {k: config[k] for k in ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob",
                                      "loss_type", "epochs", "train_batch_size", "learning_rate", "seed", "eval_args", "topk", "valid_metric")},
    "n_parameters": int(n_params),
    "n_query_parameters": int(n_query_params),
    "n_parameters_history_only": plain_ref["n_parameters"],
    "splits": splits,
    "query_coverage": coverage.to_dict(orient="index"),
    "timing_seconds": {"data_build": round(t_data, 1), "train_and_eval": round(t_fit, 1), "test": round(t_test, 1)},
    "best_epoch_by_valid": best["epoch"],
    "checkpoint": str(RUN_DIR / "best.pth"),
    "per_epoch": per_epoch,
    "valid": valid_best,
    "test": test_best,
    "plain_rescored_valid": plain_valid,
    "plain_rescored_test": plain_test,
    "heuristics_test": heuristics,
    "slices_test": {f"{a} | {b}": v for (a, b), v in slices.to_dict(orient="index").items()},
    "bootstrap_query_minus_plain_test": boot,
    "diagnostics_test": diag,
    "prior_fusion_check": {
        "formula": "score = log p_query_token + w * log p_prior(cluster | searched destination); prior as nb03 (m=5, market backoff, cut before last train target)",
        "w_grid": W_GRID,
        "valid_sweep": {str(w): v for w, v in sweep_valid.items()},
        "chosen_w_on_valid_recall@5": best_w,
        "test": fusion_test,
        "bootstrap_fused_minus_query_token_test": boot_fusion,
        "test_slices_recall@5_ndcg@5": {f"{a} | {b}": v for (a, b), v in slice_fusion.to_dict(orient="index").items()},
        "prior_n_bookings": tables["n_bookings"],
        "prior_n_destinations": len(tables["dest"]),
    },
}
result["samedest_hybrid"] = samedest_result
RESULT_JSON.parent.mkdir(parents=True, exist_ok=True)
for path in (RESULT_JSON, RUN_DIR / "metrics.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("wrote", RESULT_JSON, "and", RUN_DIR / "metrics.json")

wrote C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\smlprec_query_run.json and C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_query-1c\metrics.json
